# URL Shortener System Design (HLD + LLD)

## Problem Statement
Design a scalable URL Shortener like TinyURL or Bitly.

Example:
- Long URL -> https://www.example.com/product/iphone
- Short URL -> https://sho.rt/abX91K

---

## Functional Requirements
- Generate short URL
- Redirect short URL to original URL
- Custom aliases
- URL expiration
- Analytics

---

## Non Functional Requirements
- High availability
- Low latency
- Scalability
- Fault tolerance

---

## High Level Architecture

Client -> Load Balancer -> URL APIs / Redirect APIs -> Redis Cache -> Database

---

## APIs

POST /shorten
GET /{shortCode}

---

## Database Schema

| Column | Type |
|---|---|
| id | bigint |
| short_code | varchar |
| long_url | text |
| created_at | timestamp |
| expiry_time | timestamp |

---

## URL Generation Strategies

### Hashing
MD5(long_url)

Problem:
- collisions possible

### Counter + Base62 (Recommended)
Auto Increment ID -> Base62

Example:
125 -> cb

---

## Why Base62?
Characters:
[a-zA-Z0-9]

Total 62 characters.

Compact URLs.

---

## Redirect Flow
1. User opens short URL
2. Redirect service extracts short code
3. Check Redis cache
4. If cache miss -> DB lookup
5. Return 302 redirect

---

## Why Redis?
- in-memory
- low latency
- read heavy optimization

---

## Scaling Strategy
- Horizontal scaling
- DB sharding
- Consistent hashing

---

## Handling Hot URLs
Solutions:
- CDN
- Redis replication
- Load balancing

---

## Analytics Pipeline
Redirect Service -> Kafka -> Analytics Workers -> Analytics DB

---

## Security
- Malware detection
- Blacklist DB
- Rate limiting

---

## Failure Handling
### Redis Failure
Fallback to DB.

### DB Failure
Use replication and failover.

---

## Low Level Design (LLD)

Main Classes:
- URLController
- URLService
- URLRepository
- CacheService
- Base62Encoder
- IDGenerator

---

## URL Entity

```java
class URL {
    Long id;
    String shortCode;
    String longUrl;
    Date createdAt;
    Date expiryTime;
}
```

---

## Base62 Encoder

```java
class Base62Encoder {

    private static final String CHARS =
        "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789";

    public String encode(long id) {

        StringBuilder sb = new StringBuilder();

        while(id > 0) {
            sb.append(CHARS.charAt((int)(id % 62)));
            id /= 62;
        }

        return sb.reverse().toString();
    }
}
```

---

## URL Shortening Service

```java
class URLService {

    public String shorten(String longUrl) {

        long id = idGenerator.generate();

        String shortCode = base62.encode(id);

        repository.save(shortCode, longUrl);

        cache.put(shortCode, longUrl);

        return shortCode;
    }
}
```

---

## Final Production Architecture

Clients -> CDN -> Load Balancer -> URL APIs -> Redis Cache -> Distributed DB

---

## Final Interview Summary

Use:
- Base62 encoding
- Redis cache
- Snowflake IDs
- Distributed DB
- Kafka analytics
- CDN caching

The system is read-heavy, so caching and scalability are critical.
